In [10]:
from aviary.core import Environment, Message, Tool, ToolRequestMessage, ToolSelector
from ldp.agent import Agent
from ldp.graph import LLMCallOp
from ldp.alg import RolloutManager
import os
from helper.problem import ProblemState, examples
from helper.units import UNIT_ALIASES
from helper.tools import *
from helper.schemas import *

In [ ]:
os.environ["OPENAI_API_KEY"] = ""

In [12]:
# Helper functions
def normalize_unit(unit: str) -> str:
    return UNIT_ALIASES.get(unit.lower().strip(), unit.strip())

In [13]:
TOOLS  = {
  "convert_units":    convert_units,
  "molar_mass":         lookup_molar_mass,
  "compute_moles":      compute_moles,
  "mass_from_moles":    mass_from_moles,
  "mix_concentration":  mix_concentration,
  "dilution": dilution,
}

class ScienceEnv(Environment[ProblemState]):
    def __init__(self, problem: ProblemState):
          super().__init__()
          self.problem = problem
          self._oracle = None
        
    async def reset(self):
        self.tools = [Tool.from_function(fun) 
                      for _, fun in TOOLS.items()]
        prompt = self.problem.prompt or build_prompt(self.problem)
        print(f"[RESET] prompt: {prompt}")
        return [Message(content=prompt)], self.tools

    async def step(self, action: Message | ToolRequestMessage):
        # execute any tool calls
        obs = await self.exec_tool_calls(action)

        if isinstance(action, ToolRequestMessage):
            return obs, 0.0, False, False

        actual   = action.content.strip()
        expected = self.problem.expected.strip()
        self._oracle = expected
        reward   = float(actual == expected)

        print(f"[STEP] expected: {expected}")
        print(f"[STEP] actual:   {actual}")
        print(f"[STEP] reward:   {reward}")

        return obs, reward, True, False

In [14]:
SYSTEM = Message(
    role="system",
    content=(
        "You are a scientific assistant. Plan and execute multiple steps:\n"
        "1) Decide which function to call next.\n"
        "2) Call it with valid JSON args.\n"
        "3) Observe the result.\n"
        "4) Repeat until you have a final answer, then reply with that answer."
    ),
)

In [15]:
import json

In [16]:
# --- Step 2: Simple Agent ---
class AgentState:
    """A container for maintaining agent state across interactions."""

    def __init__(self, messages, tools):
        self.messages = messages
        self.tools = tools


class PlanningAgent(Agent):
    def __init__(self, **kwargs):
        self._llm_call_op = LLMCallOp(**kwargs)
        self.tool_selector = ToolSelector(model_name="gpt-4o", accum_messages=True)

    async def init_state(self, tools):
        print(f"[DEBUG]: {SYSTEM}")
        return AgentState(messages=[SYSTEM], tools=tools)

    async def get_asv(self, agent_state, obs):
        """Take an action, observe new state, return value"""
        # append new observation
        msgs=agent_state.messages + obs
        
        # tool selcetor
        action = await self.tool_selector(msgs, agent_state.tools)
        
        if isinstance(action, ToolRequestMessage) and action.tool_calls:
            tc = action.tool_calls[0]
            print("Agent chose tool:", tc.function.name)
            print("…with args:", tc.function.arguments)
        else:
            print("Agent final answer:", action.content)
        
        print(f"[DEBUG]: {agent_state.tools}")
        print(f"[DEBUG]: {action}")
        
        new_state = AgentState(messages=msgs + [action], tools=agent_state.tools)

        return action, new_state, 0.0

In [17]:
examples.append(
  ProblemState(
    prompt=(
      "You have 50 mL of a 0.2 M NaCl solution. "
      "If you dilute it to 0.1 M, what is the final volume in mL?"
    ),
    task="dilution",
    params={"initial_conc": 0.2, "initial_vol": 0.05, "final_conc": 0.1},
    expected="100.000 mL"
  )
)

In [18]:
# --- Step 3: Rollout ---

agent = PlanningAgent()
runner = RolloutManager(agent=agent)

for problem in examples[6:8]:
    env = ScienceEnv(problem)
    obs, tools = await env.reset()

    # initialize agent state
    agent_state = await agent.init_state(tools)

    for _ in range(5):                  # allow up to 5 chained calls
        action, agent_state, _ = await agent.get_asv(agent_state, obs)
        obs, reward, done, _ = await env.step(action)
        if done:
            print("Final answer:", obs[-1].content)
            break

    print("Oracle:", env._oracle, "Reward:", reward)



[DEBUG build_prompt] task='dilution', params={'initial_conc': 0.5, 'initial_vol': 0.1, 'final_conc': 0.1}
[RESET] prompt: If you have 0.5 M at 0.1 L, how much volume is needed to get 0.1 M?
[DEBUG]: You are a scientific assistant. Plan and execute multiple steps:
1) Decide which function to call next.
2) Call it with valid JSON args.
3) Observe the result.
4) Repeat until you have a final answer, then reply with that answer.
Agent chose tool: dilution
…with args: {'initial_conc': 0.5, 'initial_vol': 0.1, 'final_conc': 0.1}
[DEBUG]: [Tool(type='function', info=FunctionInfo(name='convert_units', description='Convert a quantity from one unit to another using Pint.', parameters=Parameters(type='object', properties={'value': {'description': 'Numeric value to convert.', 'title': 'Value', 'type': 'number'}, 'from_unit': {'description': "Source unit (e.g., 'g', 'degC').", 'title': 'From Unit', 'type': 'string'}, 'to_unit': {'description': "Target unit (e.g., 'kg', 'degF').", 'title': 'To Unit'

BadRequestError: litellm.BadRequestError: OpenAIException - An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_fpyjUMGBoDo5BVqdeX2E8bGf